## Gradient Clipping

In [1]:
optimizer.zero_grad()
loss, hidden = model(data, hidden, targets)
loss.backward()

clipping_value = 1
torch.nn.utils.clip_grad_norm_(model.parameters(), clipping_value)
optimizer.step()

## Классификация текстов

В этом задании вам предстоит попробовать несколько методов, используемых в задаче классификации, а также понять насколько хорошо модель понимает смысл слов и какие слова в примере влияют на результат.

In [2]:
import random
import re
from collections import Counter

import numpy as np
import pandas as pd
from tqdm import tqdm

from datasets import load_dataset

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
SEED = 1234
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

In [4]:
def load_imdb_data_hf():
    # Загружаем датасет через Hugging Face
    dataset = load_dataset("stanfordnlp/imdb")

    # Преобразуем в списки, как в вашем исходном коде
    train_texts = dataset["train"]["text"]
    train_labels = dataset["train"]["label"]
    test_texts = dataset["test"]["text"]
    test_labels = dataset["test"]["label"]

    return (train_texts, train_labels), (test_texts, test_labels)

In [5]:
# Загрузка данных (без параметра пути, HF сам управляет кэшем)
(train_texts, train_labels), (test_texts, test_labels) = load_imdb_data_hf()

In [6]:
def get_tokenizer():
    def tokenizer(text):
        # Простая токенизация по словам и знакам препинания
        text = text.lower()
        return re.findall(r"\b\w+\b|[^\w\s]", text)
    return tokenizer

In [7]:
def build_vocab_from_iterator(iterator, specials=None):
    counter = Counter()
    for tokens in iterator:
        counter.update(tokens)
    # Формируем словарь: specials идут первыми
    itos = list(specials) if specials else []
    itos += [word for word, _ in counter.most_common()]
    stoi = {word: i for i, word in enumerate(itos)}
    return type("Vocab", (), {"stoi": stoi, "itos": itos, "__len__": lambda self: len(itos)})()

In [8]:
tokenizer = get_tokenizer()
UNK_TOKEN, PAD_TOKEN = "<unk>", "<pad>"

def yield_tokens(texts):
    for t in texts:
        yield tokenizer(t)

vocab = build_vocab_from_iterator(
    yield_tokens(train_texts),
    specials=[UNK_TOKEN, PAD_TOKEN],
)
vocab.stoi.setdefault(UNK_TOKEN, len(vocab.stoi))
PAD_IDX = vocab.stoi[PAD_TOKEN]

def encode(text):
    return torch.tensor([vocab.stoi.get(tok, vocab.stoi[UNK_TOKEN]) for tok in tokenizer(text)], dtype=torch.long)

# --- Разделение train на train/valid ---
random.seed(SEED)
indices = list(range(len(train_texts)))
random.shuffle(indices)
split = int(0.8 * len(indices))
train_idx, valid_idx = indices[:split], indices[split:]

train_texts_split = [train_texts[i] for i in train_idx]
train_labels_split = [train_labels[i] for i in train_idx]
valid_texts = [train_texts[i] for i in valid_idx]
valid_labels = [train_labels[i] for i in valid_idx]

# --- Dataset / DataLoader ---
class IMDBDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels):
        self.texts, self.labels = texts, labels
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, i):
        return encode(self.texts[i]), torch.tensor(self.labels[i], dtype=torch.float)

def collate_batch(batch):
    # сортируем по убыванию длины — нужно для pack_padded_sequence
    batch.sort(key=lambda x: len(x[0]), reverse=True)
    texts, labels = zip(*batch)
    lengths = torch.tensor([len(t) for t in texts], dtype=torch.long)
    padded = pad_sequence(texts, batch_first=False, padding_value=PAD_IDX)
    return padded, lengths, torch.stack(labels)

train_iter = DataLoader(IMDBDataset(train_texts_split, train_labels_split), batch_size=128, shuffle=True, collate_fn=collate_batch)
valid_iter = DataLoader(IMDBDataset(valid_texts, valid_labels), batch_size=256, shuffle=False, collate_fn=collate_batch)
test_iter  = DataLoader(IMDBDataset(test_texts, test_labels), batch_size=256, shuffle=False, collate_fn=collate_batch)

device = DEVICE

## RNN

Попробуем применить рекурентные нейронные сети (GRU или LSTM).

Для задач классификации используют как hidden_state, так и output последнего токена.

In [9]:
class RNNBaseline(nn.Module):
    def __init__(
        self,
        vocab_size,
        embedding_dim,
        hidden_dim,
        output_dim,
        n_layers,
        bidirectional,
        dropout,
        pad_idx,
    ) -> None:
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers=n_layers, bidirectional=bidirectional)
        self.fc = nn.Linear(hidden_dim*2, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, text, text_lengths):

        # text = [sent_len, batch_size]
        embedded = self.embedding(text)

        # embedded = [sent_len, batch_size, emb_dim]
        # pack sequence
        packed_embedded = nn.utils.rnn.pack_padded_sequence(embedded, text_lengths.cpu(), enforce_sorted=False)

        # LSTM
        packed_output, (hidden, cell) = self.lstm(packed_embedded)

        # OUTPUT
        # output = [sent_len, batch_size, hid_dim * num_directions]
        # unpack sequence
        output, output_lengths = nn.utils.rnn.pad_packed_sequence(packed_output)
        # get last token from output
        # output = [batch_size, hid_dim * num_directions]
        output = output[-1]

        # HIDDEN
        # hidden = [num_layers * num_directions, batch_size, hid_dim]
        # cell = [num_layers * num_directions, batch_size, hid_dim]
        # get hidden from two last layers
        # hidden = [batch_size, hid_dim * num_layers]
        hidden = torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)

        out_drop = self.dropout(hidden)
        out = self.fc(out_drop)

        return out

In [10]:
vocab_size = len(vocab.stoi)
emb_dim = 100
hidden_dim = 256
output_dim = 1
n_layers = 2
bidirectional = True
dropout = 0.2
patience = 3
max_epochs = 20

In [11]:
model = RNNBaseline(
    vocab_size=vocab_size,
    embedding_dim=emb_dim,
    hidden_dim=hidden_dim,
    output_dim=output_dim,
    n_layers=n_layers,
    bidirectional=bidirectional,
    dropout=dropout,
    pad_idx=PAD_IDX
)

model = model.to(device)

In [12]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_func = nn.BCEWithLogitsLoss()
min_loss = float("inf")
cur_patience = 0

for epoch in range(1, max_epochs + 1):
    model.train()
    train_loss = 0.0
    for texts, lengths, labels in tqdm(train_iter, desc=f"Epoch {epoch}"):
        texts, lengths, labels = texts.to(device), lengths.to(device), labels.to(device)
        optimizer.zero_grad()
        preds = model(texts, lengths).squeeze()
        loss = loss_func(preds, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_iter)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for texts, lengths, labels in valid_iter:
            texts, lengths, labels = texts.to(device), lengths.to(device), labels.to(device)
            preds = model(texts, lengths).squeeze()
            val_loss += loss_func(preds, labels).item()
    val_loss /= len(valid_iter)

    print(f"Epoch {epoch}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}")
    if val_loss < min_loss:
        min_loss = val_loss
        best_state = model.state_dict()
        cur_patience = 0
    else:
        cur_patience += 1
        if cur_patience == patience:
            break

model.load_state_dict(best_state)

Epoch 1: 100%|██████████| 157/157 [00:49<00:00,  3.16it/s]


Epoch 1: train_loss=0.6133, val_loss=0.5807


Epoch 2: 100%|██████████| 157/157 [00:48<00:00,  3.24it/s]


Epoch 2: train_loss=0.5426, val_loss=0.5396


Epoch 3: 100%|██████████| 157/157 [00:48<00:00,  3.21it/s]


Epoch 3: train_loss=0.4261, val_loss=0.4954


... (13 outputs left out)

Epoch 10: train_loss=0.0546, val_loss=0.5697


Epoch 11: 100%|██████████| 157/157 [00:48<00:00,  3.21it/s]


Epoch 11: train_loss=0.0453, val_loss=0.4056


<All keys matched successfully>

Посчитаем f1-score нашего классификатора на тестовом датасете.

In [13]:
from sklearn.metrics import f1_score

all_preds = []
all_labels = []
with torch.no_grad():
    for texts, lengths, labels in test_iter:
        texts, lengths = texts.to(device), lengths.to(device)
        probs = torch.sigmoid(model(texts, lengths).squeeze()).cpu().numpy()
        all_preds.extend((probs > 0.5).astype(int))
        all_labels.extend(labels.numpy())

print("F1 (RNN):", f1_score(all_labels, all_preds))

F1 (RNN): 0.867766919427436


## CNN

![](https://www.researchgate.net/publication/333752473/figure/fig1/AS:769346934673412@1560438011375/Standard-CNN-on-text-classification.png)

Вы можете использовать Conv2d с `in_channels=1, kernel_size=(kernel_sizes[0], emb_dim))` или Conv1d c `in_channels=emb_dim, kernel_size=kernel_size[0]`.

In [14]:
class CNN(nn.Module):
    def __init__(self, vocab_size, emb_dim, out_channels, kernel_sizes, dropout=0.5) -> None:
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim)
        self.conv_0 = nn.Conv1d(emb_dim, out_channels, kernel_size=kernel_sizes[0])
        self.conv_1 = nn.Conv1d(emb_dim, out_channels, kernel_size=kernel_sizes[1])
        self.conv_2 = nn.Conv1d(emb_dim, out_channels, kernel_size=kernel_sizes[2])
        self.fc = nn.Linear(len(kernel_sizes) * out_channels, 1)
        self.dropout = nn.Dropout(dropout)

    def forward(self, text):
        embedded = self.embedding(text)

        conved_0 = F.relu(self.conv_0(embedded.permute(0,2,1)))
        conved_1 = F.relu(self.conv_1(embedded.permute(0,2,1)))
        conved_2 = F.relu(self.conv_2(embedded.permute(0,2,1)))

        pooled_0 = F.max_pool1d(conved_0, conved_0.shape[2]).squeeze(2)
        pooled_1 = F.max_pool1d(conved_1, conved_1.shape[2]).squeeze(2)
        pooled_2 = F.max_pool1d(conved_2, conved_2.shape[2]).squeeze(2)

        cat = self.dropout(torch.cat((pooled_0, pooled_1, pooled_2), dim=1))

        return self.fc(cat)

In [15]:
def collate_batch_bf(batch):
    texts, labels = zip(*batch)
    padded = pad_sequence(texts, batch_first=True, padding_value=PAD_IDX)
    return padded, torch.stack(labels)

train_iter_cnn = DataLoader(IMDBDataset(train_texts_split, train_labels_split), batch_size=128, shuffle=True, collate_fn=collate_batch_bf)
valid_iter_cnn = DataLoader(IMDBDataset(valid_texts, valid_labels), batch_size=256, shuffle=False, collate_fn=collate_batch_bf)
test_iter_cnn  = DataLoader(IMDBDataset(test_texts, test_labels), batch_size=256, shuffle=False, collate_fn=collate_batch_bf)

In [16]:
kernel_sizes = [3, 4, 5]
out_channels = 64
dim = 300
max_epochs = 20
patience = 3

In [17]:
model = CNN(
    vocab_size=vocab_size,   # vocab_size уже посчитан выше = len(vocab.stoi)
    emb_dim=dim,
    out_channels=out_channels,
    kernel_sizes=kernel_sizes,
    dropout=0.2,
)

model = model.to(device)

In [18]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_func = nn.BCEWithLogitsLoss()
min_loss = float("inf")
cur_patience = 0

for epoch in range(1, max_epochs + 1):
    model.train()
    train_loss = 0.0
    for texts, labels in tqdm(train_iter_cnn, desc=f"CNN epoch {epoch}"):
        texts, labels = texts.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = loss_func(model(texts).squeeze(), labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_iter_cnn)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for texts, labels in valid_iter_cnn:
            texts, labels = texts.to(device), labels.to(device)
            val_loss += loss_func(model(texts).squeeze(), labels).item()
    val_loss /= len(valid_iter_cnn)

    print(f"CNN epoch {epoch}: train={train_loss:.4f}, val={val_loss:.4f}")
    if val_loss < min_loss:
        min_loss = val_loss
        best_state = model.state_dict()
        cur_patience = 0
    else:
        cur_patience += 1
        if cur_patience == patience:
            break

model.load_state_dict(best_state)

CNN epoch 1: 100%|██████████| 157/157 [00:29<00:00,  5.33it/s]


CNN epoch 1: train=0.5316, val=0.3921


CNN epoch 2: 100%|██████████| 157/157 [00:29<00:00,  5.41it/s]


CNN epoch 2: train=0.3509, val=0.3345


CNN epoch 3: 100%|██████████| 157/157 [00:28<00:00,  5.42it/s]


CNN epoch 3: train=0.2774, val=0.2936


... (5 outputs left out)

CNN epoch 6: train=0.1177, val=0.2811


CNN epoch 7: 100%|██████████| 157/157 [00:29<00:00,  5.34it/s]


CNN epoch 7: train=0.0886, val=0.2925


<All keys matched successfully>

Посчитаем f1-score нашего классификатора на тестовом датасете.

In [19]:
all_preds = []
all_labels = []
with torch.no_grad():
    for texts, labels in test_iter_cnn:
        probs = torch.sigmoid(model(texts.to(device)).squeeze()).cpu().numpy()
        all_preds.extend((probs > 0.5).astype(int))
        all_labels.extend(labels.numpy())

print("F1 (CNN):", f1_score(all_labels, all_preds))

F1 (CNN): 0.8881977274612258


## Интерпретируемость

In [20]:
!pip install -q captum

In [21]:
from captum.attr import LayerIntegratedGradients, TokenReferenceBase, visualization

token_reference = TokenReferenceBase(reference_token_idx=PAD_IDX)
lig = LayerIntegratedGradients(model, model.embedding)

In [22]:
def forward_with_sigmoid(inp):
    return torch.sigmoid(model(inp))

vis_data_records_ig = []

def interpret_sentence(model, sentence, min_len=7, label=0):
    model.eval()
    tokens = tokenizer(sentence)
    if len(tokens) < min_len:
        tokens += ["<pad>"] * (min_len - len(tokens))
    indexed = [vocab.stoi.get(t, vocab.stoi[UNK_TOKEN]) for t in tokens]
    input_indices = torch.tensor(indexed, device=device).unsqueeze(0)

    pred = forward_with_sigmoid(input_indices).item()
    pred_ind = int(round(pred))

    ref = token_reference.generate_reference(len(tokens), device=device).unsqueeze(0)
    attributions, delta = lig.attribute(input_indices, ref, n_steps=50,
                                        return_convergence_delta=True)

    print(f"pred={pred:.2f} ({'pos' if pred_ind else 'neg'}), delta={abs(float(delta)):.4f}")

    attributions = attributions.sum(dim=2).squeeze(0)
    attributions = attributions / torch.norm(attributions)
    attributions = attributions.cpu().detach().numpy()

    vis_data_records_ig.append(
        visualization.VisualizationDataRecord(
            attributions, pred,
            "pos" if pred_ind else "neg",
            "pos" if label else "neg",
            "pos",
            attributions.sum(), tokens, delta,
        )
    )

In [23]:
for s, l in [
    ("It was a fantastic performance !", 1),
    ("Best film ever", 1),
    ("Such a great show!", 1),
    ("It was a horrible movie", 0),
    ("I've never watched something as bad", 0),
    ("It is a disgusting movie!", 0),
]:
    interpret_sentence(model, s, label=l)

print("Visualize attributions based on Integrated Gradients")
visualization.visualize_text(vis_data_records_ig)

pred=1.00 (pos), delta=0.0021
pred=0.93 (pos), delta=0.0049
pred=1.00 (pos), delta=0.0001
pred=0.00 (neg), delta=0.0095
pred=0.64 (pos), delta=0.0073
pred=0.91 (pos), delta=0.0043
Visualize attributions based on Integrated Gradients


True Label,Predicted Label,Attribution Label,Attribution Score,Word Importance
pos,pos (1.00),pos,1.52,it was a fantastic performance ! #pad
pos,pos (0.93),pos,1.69,best film ever #pad #pad #pad #pad
pos,pos (1.00),pos,1.63,such a great show ! #pad #pad
neg,neg (0.00),pos,-1.16,it was a horrible movie #pad #pad
neg,pos (0.64),pos,0.56,i ' ve never watched something as bad
neg,pos (0.91),pos,1.63,it is a disgusting movie ! #pad


True Label,Predicted Label,Attribution Label,Attribution Score,Word Importance
pos,pos (1.00),pos,1.52,it was a fantastic performance ! #pad
pos,pos (0.93),pos,1.69,best film ever #pad #pad #pad #pad
pos,pos (1.00),pos,1.63,such a great show ! #pad #pad
neg,neg (0.00),pos,-1.16,it was a horrible movie #pad #pad
neg,pos (0.64),pos,0.56,i ' ve never watched something as bad
neg,pos (0.91),pos,1.63,it is a disgusting movie ! #pad
